In [121]:
import torch
import einops

In [122]:
from utils.logsumexp import logsumexp_infsafe as logsumexp

In [123]:
def vector_gather(vectors, indices):
    """
    Gathers (batched) vectors according to indices.
    Arguments:
        vectors: Tensor[N, L, D]
        indices: Tensor[N, K] or Tensor[N]
    Returns:
        Tensor[N, K, D] or Tensor[N, D]
    """
    N, L, D = vectors.shape
    squeeze = False
    if indices.ndim == 1:
        squeeze = True
        indices = indices.unsqueeze(-1)
    N2, K = indices.shape
    assert N == N2
    indices = einops.repeat(indices, "N K -> N K D", D=D)
    out = torch.gather(vectors, dim=1, index=indices)
    if squeeze:
        out = out.squeeze(1)
    return out

In [124]:
def dag_loss(targets, transition_matrix, emission_probs):
    batch_size, m = targets.shape
    _, l, vocab_size = emission_probs.shape
    dp = torch.ones((batch_size, m, l))
    dp[dp == 1] = -float('inf')
    initial_probs = torch.gather(emission_probs, dim=2, index=targets[:, 0].unsqueeze(1).unsqueeze(2))
    dp[:, 0, 0] = initial_probs.squeeze(2).squeeze(1)
    # assumes that transition_matrix and emission_probs are already in log space
    # also we need to tranpose emission_probs so it is vocab_size x l
    # so the vector gather works
    emission_probs = emission_probs.transpose(1, 2)
    for i in range(1, m):
        dp[:, i, :] = vector_gather(emission_probs, targets[:, i]) + ((logsumexp(dp[:, i-1, :].unsqueeze(1).transpose(1, 2) + transition_matrix, dim=1)).squeeze(1))
    return dp

In [125]:
def process_dp(dp, target_lens, vertex_lens):
    """
    Processes the dynamic programming table (dp) to extract the correct loss values.
    The target lengths and vertex lengths are needed to determine which values to extract
    and which values are a result of padding and should be ignored.

    Args:
        dp (torch.Tensor): The dynamic programming table of shape (batch_size, m, l).
        target_lens (torch.Tensor): A tensor of shape (batch_size,) that describes the length of each target sequence.
        vertex_lens (torch.Tensor): A tensor of shape (batch_size,) that describes the number of non-padding vertices for each batch.

    Returns:
        torch.Tensor: The values corresponding to the last target and last vertex of shape (batch_size,).
    """
    dp_values = vector_gather(dp, target_lens - 1)
    values = torch.gather(dp_values, dim=1, index=(vertex_lens - 1).unsqueeze(-1))
    return values

## Test Case 1

In [126]:
transition_matrix1 = torch.tensor(
    [
        [0, 0.8, 0.1, 0.1],
        [0, 0, 0.8, 0.2],
        [0, 0, 0, 1],
        [0, 0, 0, 0]
    ]
)
emission_matrix1 = torch.tensor(
    [
        [0.9, 0.1, 0, 0],
        [0, 0.8, 0.2, 0],
        [0, 0, 0.9, 0.1],
        [0, 0.3, 0, 0.7]
    ]
)
targets = torch.tensor([0, 1, 2, 3])
target_lens1 = torch.tensor([4])
vertex_lens1 = torch.tensor([4])
expected_answer1 = torch.tensor(
    [
        transition_matrix1[0][1],
        transition_matrix1[1][2],
        transition_matrix1[2][3],
        emission_matrix1[0][0],
        emission_matrix1[1][1],
        emission_matrix1[2][2],
        emission_matrix1[3][3]
    ]
)
expected_answer1 = torch.prod(expected_answer1)

In [127]:
transition_matrix1 = torch.log(transition_matrix1)
emission_matrix1 = torch.log(emission_matrix1)

In [128]:
acyclic_mask1 = torch.tril(torch.ones((4, 4)))

In [129]:
transition_matrix1 = transition_matrix1.masked_fill(acyclic_mask1 != 0, -float('inf'))

## Test Case 2

In [130]:
transition_matrix2 = torch.tensor(
    [
        [0, 0.8, 0, 0.2],
        [0.6, 0.1, 0.2, 0.1],
        [0, 0, 0, 1],
        [0, 0, 0, 0]
    ]
)
emission_matrix2 = torch.tensor(
    [
        [0.5, 0, 0.5, 0],
        [0, 0.8, 0.2, 0],
        [0, 0, 0.9, 0.1],
        [0, 0.1, 0, 0.9]
    ]
)
targets2 = torch.tensor([0, 1, 2, 3])
target_lens2 = torch.tensor([4])
vertex_lens2 = torch.tensor([4])
expected_answer2 = torch.tensor(
    [
        transition_matrix2[0][1],
        transition_matrix2[1][2],
        transition_matrix2[2][3],
        emission_matrix2[0][0],
        emission_matrix2[1][1],
        emission_matrix2[2][2],
        emission_matrix2[3][3]
    ]
)
expected_answer2 = torch.prod(expected_answer2)

In [131]:
transition_matrix2 = torch.log(transition_matrix2)
emission_matrix2 = torch.log(emission_matrix2)

In [132]:
acycle_mask2 = torch.tril(torch.ones((4, 4)))

In [133]:
transition_matrix2 = transition_matrix2.masked_fill(acycle_mask2 != 0, -float('inf'))

## Test case 3

In [134]:
transition_matrix3 = torch.tensor(
    [
        [0, 0.8, 0, 0.2],
        [0.6, 0.1, 0.2, 0.1],
        [0, 0, 0, 1],
        [0.3, 0.3, 0.2, 0.2]
    ]
)
emission_matrix3 = torch.tensor(
    [
        [0.5, 0, 0.5, 0],
        [0, 0.8, 0.2, 0],
        [0, 0, 0.9, 0.1],
        [0, 0.1, 0, 0.9]
    ]
)
targets3 = torch.tensor([0, 1, 2, 3]) # note the final 3 is dummy due to padding
target_lens3 = torch.tensor([3])
vertex_lens3 = torch.tensor([3])
expected_answer3 = torch.tensor(
    [
        transition_matrix3[0][1],
        transition_matrix3[1][2],
        emission_matrix3[0][0],
        emission_matrix3[1][1],
        emission_matrix3[2][2],
    ]
)
expected_answer3 = torch.prod(expected_answer3)

In [135]:
transition_matrix3 = torch.log(transition_matrix3)
emission_matrix3 = torch.log(emission_matrix3)

In [136]:
acycle_mask3 = torch.tril(torch.ones((4, 4)))

In [137]:
# case 3 is special because it has padding column so we need a padding mask
# in this case, the padding mask is all False safe for the last column
padding_mask = targets3 == 3

In [138]:
padding_mask = padding_mask.repeat(4, 1)

In [139]:
mask3 = acycle_mask3.masked_fill(padding_mask, 1)

In [140]:
transition_matrix3 = transition_matrix3.masked_fill(mask3 != 0, -float('inf'))

In [141]:
transition_matrices = torch.stack([transition_matrix1, transition_matrix2, transition_matrix3])
emission_matrices = torch.stack([emission_matrix1, emission_matrix2, emission_matrix3])
targets = torch.stack([targets, targets2, targets3])
target_lens = torch.stack([target_lens1, target_lens2, target_lens3])
vertex_lens = torch.stack([vertex_lens1, vertex_lens2, vertex_lens3])
expected_answers = torch.stack([expected_answer1, expected_answer2, expected_answer3])

In [142]:
target_lens = target_lens.squeeze(-1)
vertex_lens = vertex_lens.squeeze(-1)

In [143]:
out = dag_loss(targets, transition_matrices, emission_matrices)

In [144]:
out = process_dp(out, target_lens, vertex_lens)

In [145]:
out

tensor([[-1.2368],
        [-2.9596],
        [-2.8542]])

In [146]:
expected_answers

tensor([0.2903, 0.0518, 0.0576])

In [150]:
# exp out and then flatten should give expected answers
torch.exp(out).flatten()

tensor([0.2903, 0.0518, 0.0576])